## 1. Environment and dependency check

Before running any inference, ensure all required Python packages are available in the current environment.
If this is your first run on a new machine or virtual environment, install the dependencies once.
After installation, you can skip this step in future runs unless you change environments.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from model_utils import SplitSeek, ESM_Model, featurize, extract_input_data
from Bio.PDB import PDBParser, MMCIFParser, PDBIO

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())

esm_model = ESM_Model()
esm_model.load("esm2_t33_650M_UR50D")

## 2. Parse PDB/MMCIF and residue filtering

This section defines utilities to read structural files (PDB or MMCIF), build chain sequences, and extract backbone coordinates.
Non-standard residues, ligands, and waters are filtered out to keep only amino-acid residues used by the model.
These helpers mirror the logic in `run.py` so notebook and CLI outputs remain consistent.

In [ ]:
res_d = {
    "GLY": "G", "ALA": "A", "VAL": "V", "LEU": "L", "ILE": "I",
    "PRO": "P", "PHE": "F", "TYR": "Y", "TRP": "W", "SER": "S",
    "THR": "T", "CYS": "C", "MET": "M", "ASN": "N", "GLN": "Q",
    "ASP": "D", "GLU": "E", "LYS": "K", "ARG": "R", "HIS": "H", "MSE": "M",
}

standard_residues = {
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS',
    'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
    'LEU', 'LYS', 'MET', 'PHE', 'PRO',
    'SER', 'THR', 'TRP', 'TYR', 'VAL'
}

def is_residues(resname):
    return resname in standard_residues or resname in {
        'SEP', 'TPO', 'PTR', 'TYS', 'CYX',
        'MSE', 'HIP', 'HID', 'HIE', 'HSD',
        'HSE', 'HSP', 'ASH', 'GLH', 'LYN',
        'ARN', 'CYM', 'CYD', 'CY1', 'CSD'
    }

def remove_non_residues(model):
    for chain in list(model):
        residues_to_remove = []
        for residue in chain:
            if not is_residues(residue.get_resname()):
                residues_to_remove.append(residue.id)
        for res_id in residues_to_remove:
            chain.detach_child(res_id)

def parse_pdb(path):
    name = os.path.split(path)[1].split('.')[0]
    if path.endswith('pdb'):
        p = PDBParser()
        s = p.get_structure(name, path)[0]
    elif path.endswith('cif'):
        c = MMCIFParser()
        s = c.get_structure(name, path)[0]
    else:
        raise ValueError('please input appropriate format')

    target_atoms = ['N', 'CA', 'C', 'O']
    info_d = {}
    length = 0
    remove_non_residues(s)

    for chain in s.get_chains():
        chain_seq = ''
        chain_coords = []
        chain_calres = []
        chain_id = chain.id
        for res in chain.get_residues():
            rn = res.get_resname()
            if rn in res_d:
                chain_calres.append(res)
                chain_seq += res_d[rn]
                res_xyz = []
                for atom_name in target_atoms:
                    if res.has_id(atom_name):
                        atom = res[atom_name]
                        xyz = list(atom.get_coord())
                        res_xyz.append(xyz)
                    else:
                        xyz = [np.nan, np.nan, np.nan]
                        res_xyz.append(xyz)
                chain_coords.append(res_xyz)
        info_d[chain_id] = [chain_seq, chain_coords, chain_calres]
        length += len(chain_seq)
    return name, info_d, length, s

def rewrite_pdb(pred_list, cal_res, structure, out_path):
    io = PDBIO()
    chain_list = structure.get_chains()
    sc_d = {}
    c = 0
    for chain, cr_chain in zip(chain_list, cal_res):
        for residue in chain.get_residues():
            res_info = ''.join([residue.get_resname(), str(residue.id[1])])
            res_info = '_'.join([chain.id, res_info])
            if residue in cr_chain:
                score = pred_list[c]
                c += 1
                for atom in residue.get_atoms():
                    atom.set_bfactor(score * 100)
            else:
                score = 0.5
                for atom in residue.get_atoms():
                    atom.set_bfactor(50)
            sc_d[res_info] = score
    io.set_structure(structure)
    io.save(out_path)
    return sc_d

## 3. Model construction and inference functions

Here we define the model builder and the inference routine used later in the pipeline.
The builder loads SplitSeek weights and prepares the ESM encoder (optionally reusing a preloaded instance).
The inference function featurizes inputs, runs the model several times, and averages results for stability.

In [ ]:
def build_model(model_name="weights_v1", esm_model=None):
    hidden_dim = 128
    num_encoder_layers = 3
    num_decoder_layers = 3
    num_neighbors = 48
    dropout = 0.1
    backbone_noise = 0

    aaindex1_path = Path("constant/aaindex.json")
    with open(aaindex1_path, 'r') as f:
        aaindex1 = json.load(f)

    weight_path = Path("weights") / f"{model_name}.pt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SplitSeek(
        aaindex1=aaindex1,
        node_features=hidden_dim,
        edge_features=hidden_dim,
        hidden_dim=hidden_dim,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        k_neighbors=num_neighbors,
        dropout=dropout,
        augment_eps=backbone_noise,
    )

    model.to(device)
    checkpoint = torch.load(weight_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    if esm_model is None:
        esm_model = ESM_Model()
        esm_model.load("esm2_t33_650M_UR50D")

    return model, esm_model

def run_splitseek(full_info_d, model, esm_model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_list = extract_input_data(full_info_d)
    X, S, ESM_S, mask, residue_idx, chain_encoding_all, batch_name = featurize(
        input_list, esm_model, device
    )
    probs_list = []
    for _ in range(10):
        probs = model(X, S, ESM_S, mask, residue_idx, chain_encoding_all)
        probs_list.append(probs)
    probs = torch.mean(torch.stack(probs_list), dim=0)
    probs = probs.view(probs.shape[0], probs.shape[1])
    probs = probs.cpu().data.numpy()
    return probs

## 4. Configure input and output paths

Specify where your input structures are located and where to write outputs.
You can set a single file path (one structure) or a directory (batch prediction).
Outputs will include a B-factor annotated PDB and a CSV with per-residue scores.

In [ ]:
# Single-file example:
# input_path = "examples/inputs/single/4eb0.pdb"
# Multi-file directory example:
input_path = "examples/inputs/multi"

# Output directory (created if missing)
output_dir = Path("examples/outputs/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

model_name = "weights_v1"

## 5. Run inference and export results

This section loads the model and iterates over the input structures.
For each structure, it predicts residue-level scores, writes a new PDB with scores in the B-factor field, and saves a CSV.
Progress is printed to help track batch runs.

In [ ]:
# Build input list
if os.path.isdir(input_path):
    input_pdb_list = [
        os.path.join(input_path, i)
        for i in os.listdir(input_path)
        if i.endswith('pdb') or i.endswith('cif')
    ]
else:
    input_pdb_list = [input_path]

print("inputs", len(input_pdb_list))

# Load model (reuse ESM initialized in step 1)
splitseek, esm = build_model(model_name, esm_model)

for pdb in input_pdb_list:
    full_info_d = {}
    name, info_d, length, s = parse_pdb(pdb)
    full_info_d[name] = info_d

    probs = run_splitseek(full_info_d, splitseek, esm)[0]
    cal_res = [i[2] for i in info_d.values()]

    out_name = f"{name}_pred"
    out_pdb = f"{out_name}.pdb"
    out_csv = f"{out_name}.csv"
    pdb_path = output_dir / out_pdb
    csv_path = output_dir / out_csv

    sc_info = rewrite_pdb(probs, cal_res, s, str(pdb_path))
    df = pd.DataFrame.from_dict(sc_info, orient='index', columns=['score'])
    df.to_csv(csv_path)
    print("done", pdb_path.name, csv_path.name)

## 6. Preview results (optional)

Optionally preview the first few rows of any output CSV to confirm scores were written correctly.
This is a quick sanity check before downstream analysis or visualization.

In [ ]:
# Preview any CSV
preview_files = sorted(output_dir.glob('*.csv'))
if preview_files:
    display(pd.read_csv(preview_files[0]).head())
else:
    print('no csv output')